# 02 — Treino dos 6 experimentos no Google Colab (GPU T4)

**Antes de rodar**
1. *Ambiente de execução → Alterar o tipo de ambiente de execução → **GPU T4***.
2. No Google Drive: `MyDrive/trashnet-mlp-vs-cnn/data/dataset-resized.zip`
   (se não estiver lá, a célula 4 baixa do Hugging Face).

**O que este notebook faz**
- Clona o repositório público (o código vem do GitHub; nada é enviado de volta pelo Colab).
- Copia o zip para o disco local do Colab e descompacta.
- Roda cada experimento em um processo Python separado (`python -m src.train`), na ordem 3 → 1 → 4 → 2 → 5 → 6.
  Processo separado = medição limpa de tempo, RAM e VRAM por experimento.
- Salva tudo no Drive em `MyDrive/trashnet-mlp-vs-cnn/outputs/<experimento>/`.
  **Se a sessão cair**, é só rodar de novo: experimento com `metrics.json` é pulado.
- No fim gera `resultados_leves.zip` no Drive (tudo menos os modelos) para colocar em `results/` no repositório.

## 1. GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 2. Configuração

In [ ]:
import os
REPO_URL = "https://github.com/LeoBat00/trashnet-mlp-vs-cnn.git"
REPO_DIR = "/content/trashnet-mlp-vs-cnn"
DRIVE_BASE = "/content/drive/MyDrive/trashnet-mlp-vs-cnn"
DATA_LOCAL = "/content/data"
ZIP_DRIVE = f"{DRIVE_BASE}/data/dataset-resized.zip"
ZIP_URL = "https://huggingface.co/datasets/garythung/trashnet/resolve/main/dataset-resized.zip?download=true"
OUTPUTS = f"{DRIVE_BASE}/outputs"

## 3. Código (clone ou atualização) e Drive

In [ ]:
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}
!git -C {REPO_DIR} log --oneline -1

from google.colab import drive
drive.mount("/content/drive")
os.makedirs(OUTPUTS, exist_ok=True)

## 4. Dados no disco local do Colab (fora da medição de tempo)

In [ ]:
if not os.path.isdir(f"{DATA_LOCAL}/dataset-resized"):
    os.makedirs(DATA_LOCAL, exist_ok=True)
    if os.path.exists(ZIP_DRIVE):
        !cp "{ZIP_DRIVE}" {DATA_LOCAL}/dataset-resized.zip
    else:
        print("Zip não encontrado no Drive; baixando do Hugging Face.")
        !wget -q -O {DATA_LOCAL}/dataset-resized.zip "{ZIP_URL}"
    !unzip -q {DATA_LOCAL}/dataset-resized.zip -d {DATA_LOCAL}
!ls {DATA_LOCAL}/dataset-resized

## 5. Ambiente e verificação da divisão

In [ ]:
import sys
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
os.environ["TRASHNET_DATA_DIR"] = DATA_LOCAL
os.environ["TRASHNET_OUTPUTS_DIR"] = OUTPUTS
# os.environ["TRASHNET_DETERMINISMO"] = "0"   # só se o TensorFlow reclamar de operação não determinística

!python -m src.split --verificar
!python -c "import tensorflow as tf, keras, sklearn, numpy; print('TF', tf.__version__, '| Keras', keras.__version__, '| sklearn', sklearn.__version__, '| numpy', numpy.__version__); print(tf.config.list_physical_devices('GPU'))"

## 6. Treinos

Uma célula por experimento. Cada uma pode ser rodada de novo sem perder o que já terminou.

### exp3_cnn_128x96 — CNN 128×96, sem pesos (primeiro, para validar o pipeline)

In [ ]:
!python -m src.train --exp exp3_cnn_128x96

### exp1_mlp_128x96 — MLP 128×96, sem pesos

In [ ]:
!python -m src.train --exp exp1_mlp_128x96

### exp4_cnn_128x96_pesos — CNN 128×96, com pesos de classe

In [ ]:
!python -m src.train --exp exp4_cnn_128x96_pesos

### exp2_mlp_128x96_pesos — MLP 128×96, com pesos de classe

In [ ]:
!python -m src.train --exp exp2_mlp_128x96_pesos

### exp5_cnn_256x192 — CNN 256×192, sem pesos

In [ ]:
!python -m src.train --exp exp5_cnn_256x192

### exp6_cnn_512x384 — CNN 512×384, sem pesos (o mais caro, por último)

In [ ]:
!python -m src.train --exp exp6_cnn_512x384

## 7. Resumo

In [ ]:
import pandas as pd
from src.report import tabela_resumo
pd.set_option("display.precision", 4)
resumo = tabela_resumo(OUTPUTS)
resumo

## 8. Exportar arquivos leves para o repositório

In [ ]:
from src.report import exportar_leves
zip_leve = exportar_leves(OUTPUTS, f"{DRIVE_BASE}/resultados_leves.zip")
print("Gerado:", zip_leve)
!unzip -l "{zip_leve}" | tail -3

**Depois daqui (no PC):**
1. Baixar `MyDrive/trashnet-mlp-vs-cnn/resultados_leves.zip` e descompactar **dentro de `results/`** do repositório
   (fica `results/experimentos/<experimento>/...`).
2. Os modelos (`model.keras`) ficam em `MyDrive/trashnet-mlp-vs-cnn/outputs/`; compartilhar essa pasta como
   *qualquer pessoa com o link* e colocar o link no README.